<div style="text-align:center;padding:14px 0 6px 0;"><span style="display:inline-block;background:linear-gradient(90deg,#ce93d8,#8e24aa,#5e35b1,#8e24aa,#ce93d8);background-size:300% 100%;-webkit-background-clip:text;background-clip:text;color:transparent;-webkit-text-fill-color:transparent;font-family:'Montserrat','Segoe UI',sans-serif;font-size:40px;font-weight:900;animation:tp4Flow 7s ease-in-out infinite;padding:10px 26px;border-bottom:3px solid #8e24aa;animation:tp4Flow 6s ease-in-out infinite,tp4Glow 3s ease-in-out infinite;">PARTIE 3 · Robustesse & MLOps</span></div>
<style>@keyframes tp4Flow{0%,100%{background-position:0% 50%}50%{background-position:100% 50%}}@keyframes tp4Glow{0%,100%{filter:brightness(1)}50%{filter:brightness(1.18)}}</style>

<div style="font-family:'Segoe UI',system-ui,sans-serif;max-width:960px;margin:auto;">
<span style="display:inline-block;background:linear-gradient(90deg,#ce93d8,#8e24aa,#5e35b1,#8e24aa,#ce93d8);background-size:300% 100%;-webkit-background-clip:text;background-clip:text;color:transparent;-webkit-text-fill-color:transparent;font-family:'Montserrat','Segoe UI',sans-serif;font-size:2.3rem;font-weight:900;animation:tp4Flow 7s ease-in-out infinite;padding:6px 0;border-bottom:3px solid #8e24aa;">TP4 : Classification Robuste — Partie 3</span>
<p style="font-size:1.08rem;margin:12px 0 14px 0;opacity:.9;">Phase III : évaluation de tous les modèles (déséquilibre, courbes Precision-Recall), robustesse face aux labels incertains, et MLOps avec MLflow.</p>
<div style="display:flex;flex-wrap:wrap;gap:12px;"><div style="flex:1 1 220px;background:rgba(142,36,170,.08);border-left:4px solid #8e24aa;border-radius:10px;padding:10px 14px;"><b style="color:#8e24aa;">Données</b><br>PneumoniaMNIST — classe positive : Pneumonie</div><div style="flex:1 1 220px;background:rgba(142,36,170,.08);border-left:4px solid #8e24aa;border-radius:10px;padding:10px 14px;"><b style="color:#8e24aa;">Bruit simulé</b><br>inversion de labels de la classe minoritaire (Normal)</div><div style="flex:1 1 220px;background:rgba(142,36,170,.08);border-left:4px solid #8e24aa;border-radius:10px;padding:10px 14px;"><b style="color:#8e24aa;">Traçabilité</b><br>MLflow (<code>mlruns/</code> dans le projet)</div><div style="flex:1 1 220px;background:rgba(142,36,170,.08);border-left:4px solid #8e24aa;border-radius:10px;padding:10px 14px;"><b style="color:#8e24aa;">Réalisé par</b><br>FOFACK ALEMDJOU Henri Joël <span style='opacity:.7'>(22P231)</span><br>ELOUNDOU NGOUMA Thomas Christian <span style='opacity:.7'>(22P294)</span><br>WATON JEUTA Ivana <span style='opacity:.7'>(24P763)</span><br>WAFO TEGUO Vitric Valère <span style='opacity:.7'>(22P446)</span><br>ZUCHUON KOUGANG Nathanael Michel <span style='opacity:.7'>(22P218)</span><br>AYISSI NGONO Odile Marie Ange <span style='opacity:.7'>(24P764)</span></div></div>
</div>
<style>@keyframes tp4Flow{0%,100%{background-position:0% 50%}50%{background-position:100% 50%}}@keyframes tp4Glow{0%,100%{filter:brightness(1)}50%{filter:brightness(1.18)}}</style>

<div style="margin-top:10px;"><span style="display:inline-block;background:linear-gradient(90deg,#ce93d8,#8e24aa,#5e35b1,#8e24aa,#ce93d8);background-size:300% 100%;-webkit-background-clip:text;background-clip:text;color:transparent;-webkit-text-fill-color:transparent;font-family:'Montserrat','Segoe UI',sans-serif;font-size:2rem;font-weight:900;animation:tp4Flow 7s ease-in-out infinite;padding-bottom:4px;border-bottom:2px solid rgba(142,36,170,.45);">🛡️ Phase III : Robustesse, Évaluation et MLOps</span></div>
<style>@keyframes tp4Flow{0%,100%{background-position:0% 50%}50%{background-position:100% 50%}}@keyframes tp4Glow{0%,100%{filter:brightness(1)}50%{filter:brightness(1.18)}}</style>

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, roc_curve, precision_recall_curve, roc_auc_score,
                             average_precision_score)

import tp4_utils as U

U.set_seed(); U.setup_plots()
data = U.load_pneumonia()
X_train, y_train, X_val, y_val, X_test, y_test = (data[k] for k in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test"])
scaler = StandardScaler().fit(X_train)
Xtr, Xva, Xte = scaler.transform(X_train), scaler.transform(X_val), scaler.transform(X_test)
P = {**U.load_results("phase1_params.json"), **U.load_results("phase2_params.json")}
print("✅ Données et hyperparamètres (choisis sur la validation en Parties 1-2) chargés :")
P

<div style="background:linear-gradient(135deg,rgba(142,36,170,.10),rgba(41,98,255,.07) 60%,rgba(79,172,254,.06));border-left:5px solid #8e24aa;border-radius:12px;padding:4px 20px 10px 20px;margin:8px 0;box-shadow:0 4px 16px rgba(142,36,170,.10);font-family:'Segoe UI',system-ui,sans-serif;line-height:1.6;">
<div style="height:3px;margin:0 -20px 6px -20px;border-radius:3px;background:linear-gradient(90deg,#8e24aa,#2962ff,#4facfe,#8e24aa);background-size:300% 100%;animation:tp4Flow 7s ease-in-out infinite;"></div>

Tous les modèles des Parties 1 et 2 sont reconstruits avec les hyperparamètres sélectionnés sur la validation.
Pour la Task 3.2, chaque famille possède aussi une **variante robuste**, obtenue par **plus de régularisation** /
**contrôle de complexité**, comme demandé dans l'énoncé :

| Famille | Baseline (Parties 1-2) | Variante robuste |
|---|---|---|
| Régression logistique (sklearn) | $C$ choisi sur val. | $C/10$ (L2 plus forte) |
| Régression logistique SGA (Partie 1) | $\lambda$ choisi sur val. | $10\lambda$ |
| SVM linéaire / RBF | $C$ choisi sur val. | $C/10$ (marge plus large, plus tolérante) |
| Arbre | profondeur choisie sur val. | profondeur $-2$, ≥ 20 ex. par feuille |
| AdaBoost | arbres de profondeur 2, $\eta=0{,}5$ | souches (profondeur 1), $\eta=0{,}1$ |
| Gradient boosting | feuilles, $\eta$ choisis sur val. | 7 feuilles, $\eta/2$, L2 = 10, ≥ 50 ex. par feuille |
| KNN | $K$ choisi sur val. | $3K$ (vote plus lissé) |

</div>
<style>@keyframes tp4Flow{0%,100%{background-position:0% 50%}50%{background-position:100% 50%}}@keyframes tp4Glow{0%,100%{filter:brightness(1)}50%{filter:brightness(1.18)}}</style>

In [ ]:
def make_model(name, robust=False):
    if name == "LogReg-SGA (ours)":
        return U.LogisticRegressionSGA(lr=P["sga_alpha0"], l2=P["sga_lambda"] * (10 if robust else 1), epochs=30)
    if name == "LogReg":
        return LogisticRegression(C=P["logreg_C"] / (10 if robust else 1), max_iter=3000)
    if name == "Gaussian NB":
        return GaussianNB()
    if name == "KNN":
        return KNeighborsClassifier(n_neighbors=P["knn_K"] * (3 if robust else 1))
    if name == "Decision tree":
        return DecisionTreeClassifier(max_depth=max(2, P["tree_depth"] - 2) if robust else P["tree_depth"],
                                      min_samples_leaf=20 if robust else 5, random_state=U.SEED)
    if name == "AdaBoost":
        return AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1 if robust else 2),
                                  n_estimators=P["ada_n_estimators"], learning_rate=0.1 if robust else 0.5, random_state=U.SEED)
    if name == "Gradient boosting":
        if robust:
            return HistGradientBoostingClassifier(learning_rate=P["hgb_learning_rate"] / 2, max_leaf_nodes=7, max_iter=P["hgb_max_iter"],
                                                  l2_regularization=10.0, min_samples_leaf=50, early_stopping=False, random_state=U.SEED)
        return HistGradientBoostingClassifier(learning_rate=P["hgb_learning_rate"], max_leaf_nodes=P["hgb_max_leaf_nodes"],
                                              max_iter=P["hgb_max_iter"], l2_regularization=1.0, early_stopping=False, random_state=U.SEED)
    if name == "Linear SVM":
        return LinearSVC(C=P["linsvm_C"] / (10 if robust else 1), loss="hinge", dual=True, max_iter=50_000, random_state=U.SEED)
    if name == "RBF-SVM":
        return SVC(kernel="rbf", C=P["rbf_C"] / (10 if robust else 1), gamma=P["rbf_gamma"])
    raise ValueError(name)

MODEL_NAMES = ["LogReg-SGA (ours)", "LogReg", "Gaussian NB", "KNN", "Decision tree", "AdaBoost", "Gradient boosting", "Linear SVM", "RBF-SVM"]
COLOR = dict(zip(MODEL_NAMES, [U.COLORS[k] for k in ["sga", "logreg", "nb", "knn", "tree", "ada", "gb", "svm_lin", "svm_rbf"]]))

<div style="margin-top:6px;"><span style="display:inline-block;background:linear-gradient(90deg,#4facfe,#2962ff,#7c4dff,#2962ff,#4facfe);background-size:300% 100%;-webkit-background-clip:text;background-clip:text;color:transparent;-webkit-text-fill-color:transparent;font-family:'Montserrat','Segoe UI',sans-serif;font-size:1.55rem;font-weight:800;animation:tp4Flow 7s ease-in-out infinite;">📊 Task 3.1 : Évaluation de tous les modèles et déséquilibre</span></div>
<style>@keyframes tp4Flow{0%,100%{background-position:0% 50%}50%{background-position:100% 50%}}@keyframes tp4Glow{0%,100%{filter:brightness(1)}50%{filter:brightness(1.18)}}</style>

<div style="background:linear-gradient(135deg,rgba(142,36,170,.10),rgba(41,98,255,.07) 60%,rgba(79,172,254,.06));border-left:5px solid #8e24aa;border-radius:12px;padding:4px 20px 10px 20px;margin:8px 0;box-shadow:0 4px 16px rgba(142,36,170,.10);font-family:'Segoe UI',system-ui,sans-serif;line-height:1.6;">
<div style="height:3px;margin:0 -20px 6px -20px;border-radius:3px;background:linear-gradient(90deg,#8e24aa,#2962ff,#4facfe,#8e24aa);background-size:300% 100%;animation:tp4Flow 7s ease-in-out infinite;"></div>

Classe **positive = Pneumonie** (la maladie à détecter). Avec $TP, FP, FN, TN$ :

$$\text{Precision}=\frac{TP}{TP+FP}\qquad \text{Recall (sensibilité)}=\frac{TP}{TP+FN}\qquad
\text{Spécificité}=\frac{TN}{TN+FP}\qquad F_1=\frac{2\,PR}{P+R}$$

Les intervalles de confiance à 95 % sont obtenus par **bootstrap** du jeu de test (1000 rééchantillonnages) :
avec 624 images de test, des écarts de 1 point entre modèles ne sont généralement **pas significatifs**.

</div>
<style>@keyframes tp4Flow{0%,100%{background-position:0% 50%}50%{background-position:100% 50%}}@keyframes tp4Glow{0%,100%{filter:brightness(1)}50%{filter:brightness(1.18)}}</style>

In [ ]:
t0 = time.perf_counter()
fitted = {n: make_model(n).fit(Xtr, y_train) for n in MODEL_NAMES}
print(f"{len(fitted)} modèles entraînés en {time.perf_counter() - t0:.0f}s")

rows, test_scores = [], {}
for n, m in fitted.items():
    s, p = U.scores(m, Xte), m.predict(Xte)
    test_scores[n] = s
    met = U.binary_metrics(y_test, p, s)
    lo, hi = U.bootstrap_ci(y_test, p, None, "f1", n_boot=1000)
    rows.append({"Model": n, **met, "f1_lo": lo, "f1_hi": hi,
                 "ap_normal": average_precision_score(1 - y_test, -s),
                 "val_acc": accuracy_score(y_val, m.predict(Xva))})
res = pd.DataFrame(rows)
U.save_results("phase3_clean_test.csv", res)
show = res.set_index("Model")[["accuracy", "precision", "recall", "specificity", "f1", "f1_lo", "f1_hi", "roc_auc", "ap", "ap_normal"]]
show.round(3)

In [ ]:
tab = res[["Model", "accuracy", "precision", "recall", "specificity", "f1", "roc_auc", "ap"]].copy()
tab["F1 [95% CI]"] = [f"{f:.3f} [{l:.3f}, {h:.3f}]" for f, l, h in zip(res.f1, res.f1_lo, res.f1_hi)]
tab = tab[["Model", "accuracy", "precision", "recall", "specificity", "F1 [95% CI]", "roc_auc", "ap"]]
tab.columns = ["Model", "Acc.", "Prec.", "Recall", "Spec.", "F1 [95\\% CI]", "ROC-AUC", "AP"]
U.save_latex_table(tab, "tab_clean", caption="Test performance of all classical models trained on clean labels "
                   "(positive class: pneumonia; 95\\% bootstrap confidence interval for F1). Best value per column in bold.",
                   label="tab:clean", bold_max=["Acc.", "Prec.", "Recall", "Spec.", "ROC-AUC", "AP"])
print("Table LaTeX écrite dans article/tables/tab_clean.tex")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
for n in MODEL_NAMES:
    s = test_scores[n]
    fpr, tpr, _ = roc_curve(y_test, s)
    pr, rc, _ = precision_recall_curve(y_test, s)
    ax[0].plot(fpr, tpr, lw=1.6, color=COLOR[n], label=f"{n} (AUC={roc_auc_score(y_test, s):.3f})")
    ax[1].plot(rc, pr, lw=1.6, color=COLOR[n], label=f"{n} (AP={average_precision_score(y_test, s):.3f})")
ax[0].plot([0, 1], [0, 1], "k:", lw=1)
ax[0].set(xlabel="false positive rate", ylabel="true positive rate (recall)", title="ROC curves")
ax[1].axhline(y_test.mean(), color="k", ls=":", lw=1, label=f"no-skill (prevalence = {y_test.mean():.2f})")
ax[1].set(xlabel="recall", ylabel="precision", title="Precision-Recall curves (positive: pneumonia)", ylim=(0.55, 1.01))
for a in ax: a.legend(fontsize=7.5, loc="lower right" if a is ax[0] else "lower left")
plt.tight_layout()
U.save_fig(fig, "fig_roc_pr")
plt.show()

<div style="background:linear-gradient(135deg,rgba(142,36,170,.10),rgba(41,98,255,.07) 60%,rgba(79,172,254,.06));border-left:5px solid #8e24aa;border-radius:12px;padding:4px 20px 10px 20px;margin:8px 0;box-shadow:0 4px 16px rgba(142,36,170,.10);font-family:'Segoe UI',system-ui,sans-serif;line-height:1.6;">
<div style="height:3px;margin:0 -20px 6px -20px;border-radius:3px;background:linear-gradient(90deg,#8e24aa,#2962ff,#4facfe,#8e24aa);background-size:300% 100%;animation:tp4Flow 7s ease-in-out infinite;"></div>

<div style="margin:10px 0 2px 0;"><span style="display:inline-block;background:linear-gradient(90deg,#4facfe,#2962ff,#7c4dff,#2962ff,#4facfe);background-size:300% 100%;-webkit-background-clip:text;background-clip:text;color:transparent;-webkit-text-fill-color:transparent;font-family:'Montserrat','Segoe UI',sans-serif;font-size:1.15rem;font-weight:800;animation:tp4Flow 7s ease-in-out infinite;">Pourquoi la courbe PR est préférable à la ROC sur données déséquilibrées</span></div>


* Le **FPR** de la ROC divise les faux positifs par le nombre (énorme) de négatifs : quand la maladie est **rare**,
  des centaines de fausses alertes ne déplacent presque pas la courbe ROC, alors qu'elles noient les vrais positifs.
* La **précision** compare les faux positifs aux vrais positifs : elle reflète directement ce que vit le clinicien
  (« parmi les alertes, combien sont réelles ? »). La ligne de base d'une courbe PR est la **prévalence**, et non 0,5.
* Autrement dit : la ROC est **invariante à la prévalence**, la PR **ne l'est pas** — c'est voulu.

Dans PneumoniaMNIST, la pneumonie est majoritaire (~62 % du test). Pour reproduire le cas typique d'un **dépistage**
(maladie rare), on garde **le même classifieur** et on **sous-échantillonne les pneumonies** du test pour faire varier
la prévalence de 50 % à 5 %. Le rapport ci-dessus inclut aussi l'AP calculée avec la classe minoritaire **Normal** comme positive (`ap_normal`).

</div>
<style>@keyframes tp4Flow{0%,100%{background-position:0% 50%}50%{background-position:100% 50%}}@keyframes tp4Glow{0%,100%{filter:brightness(1)}50%{filter:brightness(1.18)}}</style>

In [ ]:
best_name = res.sort_values("val_acc", ascending=False).iloc[0].Model     # meilleur modèle en validation
s_best = test_scores[best_name]
neg, pos = np.flatnonzero(y_test == 0), np.flatnonzero(y_test == 1)
rng = np.random.default_rng(U.SEED)
prevs, rows = [0.5, 0.3, 0.2, 0.1, 0.05], []
fig, ax = plt.subplots(1, 2, figsize=(11, 3.9))
cmap = plt.get_cmap("plasma")
for k, pi in enumerate(prevs):
    n_pos = int(round(pi * len(neg) / (1 - pi)))
    aucs, aps = [], []
    for r in range(200):
        idx = np.r_[neg, rng.choice(pos, n_pos, replace=False)]
        aucs.append(roc_auc_score(y_test[idx], s_best[idx])); aps.append(average_precision_score(y_test[idx], s_best[idx]))
        if r == 0:
            fpr, tpr, _ = roc_curve(y_test[idx], s_best[idx]); pr, rc, _ = precision_recall_curve(y_test[idx], s_best[idx])
            ax[0].plot(fpr, tpr, color=cmap(k / len(prevs)), lw=1.6, label=f"prevalence {pi:.0%}")
            ax[1].plot(rc, pr, color=cmap(k / len(prevs)), lw=1.6, label=f"prevalence {pi:.0%}")
    rows.append({"prevalence": pi, "n_pos": n_pos, "roc_auc_mean": np.mean(aucs), "roc_auc_std": np.std(aucs),
                 "ap_mean": np.mean(aps), "ap_std": np.std(aps)})
ax[0].set(xlabel="FPR", ylabel="TPR", title=f"ROC of {best_name}: unchanged by prevalence")
ax[1].set(xlabel="recall", ylabel="precision", title=f"PR of {best_name}: collapses when disease is rare")
for a in ax: a.legend(fontsize=8)
plt.tight_layout()
U.save_fig(fig, "fig_prevalence")
plt.show()
prev_df = pd.DataFrame(rows)
U.save_results("prevalence_study.csv", prev_df)
prev_df.round(3)

<div style="margin-top:6px;"><span style="display:inline-block;background:linear-gradient(90deg,#4facfe,#2962ff,#7c4dff,#2962ff,#4facfe);background-size:300% 100%;-webkit-background-clip:text;background-clip:text;color:transparent;-webkit-text-fill-color:transparent;font-family:'Montserrat','Segoe UI',sans-serif;font-size:1.55rem;font-weight:800;animation:tp4Flow 7s ease-in-out infinite;">🎯 Task 3.2 : Robustesse face à l'incertitude des labels</span></div>
<style>@keyframes tp4Flow{0%,100%{background-position:0% 50%}50%{background-position:100% 50%}}@keyframes tp4Glow{0%,100%{filter:brightness(1)}50%{filter:brightness(1.18)}}</style>

<div style="background:linear-gradient(135deg,rgba(142,36,170,.10),rgba(41,98,255,.07) 60%,rgba(79,172,254,.06));border-left:5px solid #8e24aa;border-radius:12px;padding:4px 20px 10px 20px;margin:8px 0;box-shadow:0 4px 16px rgba(142,36,170,.10);font-family:'Segoe UI',system-ui,sans-serif;line-height:1.6;">
<div style="height:3px;margin:0 -20px 6px -20px;border-radius:3px;background:linear-gradient(90deg,#8e24aa,#2962ff,#4facfe,#8e24aa);background-size:300% 100%;animation:tp4Flow 7s ease-in-out infinite;"></div>

**Protocole (énoncé)** : inverser **5 % des labels d'entraînement de la classe minoritaire** (Normal → Pneumonie,
soit une radiographie saine annotée « malade »), ré-entraîner les **trois meilleurs modèles**, et comparer la chute de
performance des modèles *baseline* et de leurs variantes *robustes*. Comme le suggère l'énoncé (*Boosted Tree, Kernel
SVM, LogReg with regularization*), on retient le **meilleur modèle (accuracy de validation) de chacune de ces trois
familles** : cela évite de retenir deux variantes du même algorithme (AdaBoost et gradient boosting sont au coude à coude).

**Corrections méthodologiques** par rapport à une comparaison naïve :

1. **Plan factoriel 2×2** : {baseline, robuste} × {labels propres, labels bruités}. On mesure la chute
   $\Delta = \text{score}_{\text{bruité}} - \text{score}_{\text{propre}}$ **pour chaque configuration** ; comparer un modèle
   baseline sur données propres à un modèle robuste sur données bruitées mélangerait deux effets.
2. **10 tirages** du bruit (graines différentes) : moyenne ± écart-type, car 5 % de la classe minoritaire ne représente que ~61 labels.
3. **Balayage du taux de bruit** (0 → 40 %) pour voir à partir d'où la régularisation fait une différence.
4. Le test reste **propre** : on mesure la capacité à apprendre la vraie fonction malgré des labels faux.
5. **Troisième variante, « seuil recalibré »** : le modèle *baseline* entraîné sur labels bruités, dont on ne change que
   le **seuil de décision**, choisi pour maximiser l'accuracy équilibrée sur le petit jeu de **validation** (supposé
   propre, vérifié par un expert : hypothèse classique du *trusted set*). Aucune ré-estimation des paramètres.

Ce bruit est **asymétrique** : il pousse le modèle à déclarer « Pneumonie » à tort ⇒ on suit la **spécificité**
(rappel de la classe Normal) en plus de l'accuracy, du F1 et de l'AUC.

</div>
<style>@keyframes tp4Flow{0%,100%{background-position:0% 50%}50%{background-position:100% 50%}}@keyframes tp4Glow{0%,100%{filter:brightness(1)}50%{filter:brightness(1.18)}}</style>

In [ ]:
ranking = res.sort_values("val_acc", ascending=False, kind="stable")[["Model", "val_acc"]].reset_index(drop=True)
print("Classement de tous les modèles sur la validation :")
print(ranking.round(4).to_string())
def best_of(family):
    return ranking[ranking.Model.isin(family)].iloc[0].Model    # en cas d'égalité, le premier de la liste
top3 = [best_of(["Gradient boosting", "AdaBoost"]), best_of(["RBF-SVM"]), best_of(["LogReg", "Linear SVM"])]
print()
print("Modèles retenus (meilleur de chaque famille : arbre boosté, SVM à noyau, linéaire régularisé) :", top3)
n_min = int((y_train == U.minority_class(y_train)).sum())
y_demo, flipped_demo = U.flip_minority_labels(y_train, 0.05, seed=0)
print(f"Classe minoritaire : {U.CLASS_NAMES[U.minority_class(y_train)]} ({n_min} images) → {len(flipped_demo)} labels inversés à 5 %")
print(f"Distribution train : propre {np.bincount(y_train)} → bruitée {np.bincount(y_demo)}")

def recalibrated_threshold(model):
    # seuil sur le score qui maximise l'accuracy équilibrée sur la validation (propre)
    s = U.scores(model, Xva)
    cands = np.unique(np.quantile(s, np.linspace(0.01, 0.99, 197)))
    return cands[int(np.argmax([balanced_accuracy_score(y_val, (s >= t).astype(int)) for t in cands]))]

def run_one(name, robust, rate, seed):
    y_noisy = y_train if rate == 0 else U.flip_minority_labels(y_train, rate, seed=seed)[0]
    m = make_model(name, robust).fit(Xtr, y_noisy)
    s_te = U.scores(m, Xte)
    key = {"model": name, "noise_rate": rate, "seed": seed}
    out = [{**key, "config": "robust" if robust else "baseline", **U.binary_metrics(y_test, m.predict(Xte), s_te)}]
    if not robust:
        thr = recalibrated_threshold(m)
        out.append({**key, "config": "recalibrated", **U.binary_metrics(y_test, (s_te >= thr).astype(int), s_te)})
    return out

def run_all(jobs):
    return pd.DataFrame([r for rs in Parallel(n_jobs=-1)(delayed(run_one)(*j) for j in jobs) for r in rs])

t0 = time.perf_counter()
jobs = [(n, r, 0.0, 0) for n in top3 for r in (False, True)]
jobs += [(n, r, 0.05, s) for n in top3 for r in (False, True) for s in range(10)]
noise5 = run_all(jobs)
print(f"{len(jobs)} entraînements en {time.perf_counter() - t0:.0f}s")

In [ ]:
metrics = ["accuracy", "f1", "specificity", "roc_auc"]
clean = noise5[noise5.noise_rate == 0].set_index(["model", "config"])[metrics]
noisy_mean = noise5[noise5.noise_rate == 0.05].groupby(["model", "config"])[metrics].mean()
noisy_std = noise5[noise5.noise_rate == 0.05].groupby(["model", "config"])[metrics].std()
summary = pd.concat({"propre": clean, "bruité (moy.)": noisy_mean, "bruité (écart-type)": noisy_std,
                     "Δ (bruité − propre)": noisy_mean - clean}, axis=1)
U.save_results("noise5_runs.csv", noise5)
summary.round(4)

In [ ]:
rows = []
for (model, config) in clean.index:
    r = {"Model": model, "Config.": config}
    for met, lab in [("accuracy", "Acc."), ("specificity", "Spec."), ("roc_auc", "AUC")]:
        c, mu, sd = clean.loc[(model, config), met], noisy_mean.loc[(model, config), met], noisy_std.loc[(model, config), met]
        r[f"{lab} clean"] = f"{c:.3f}"
        r[f"{lab} noisy"] = f"{mu:.3f}$\\pm${sd:.3f}"
        r[f"$\\Delta${lab}"] = f"{mu - c:+.3f}"
    rows.append(r)
U.save_latex_table(pd.DataFrame(rows), "tab_noise5",
                   caption="Robustness to flipping 5\\% of the minority-class (normal) training labels. \\emph{baseline}: hyper-parameters "
                           "selected on validation; \\emph{robust}: stronger regularisation; \\emph{recalibrated}: baseline with the "
                           "decision threshold re-selected on the clean validation set. Clean: single run; noisy: mean$\\pm$std "
                           "over 10 random flips. $\\Delta$ = noisy $-$ clean (the test set is always clean).",
                   label="tab:noise5", column_format="ll" + "ccc" * 3)
print("Table LaTeX écrite dans article/tables/tab_noise5.tex")

<div style="background:linear-gradient(135deg,rgba(142,36,170,.10),rgba(41,98,255,.07) 60%,rgba(79,172,254,.06));border-left:5px solid #8e24aa;border-radius:12px;padding:4px 20px 10px 20px;margin:8px 0;box-shadow:0 4px 16px rgba(142,36,170,.10);font-family:'Segoe UI',system-ui,sans-serif;line-height:1.6;">
<div style="height:3px;margin:0 -20px 6px -20px;border-radius:3px;background:linear-gradient(90deg,#8e24aa,#2962ff,#4facfe,#8e24aa);background-size:300% 100%;animation:tp4Flow 7s ease-in-out infinite;"></div>

<div style="margin:10px 0 2px 0;"><span style="display:inline-block;background:linear-gradient(90deg,#4facfe,#2962ff,#7c4dff,#2962ff,#4facfe);background-size:300% 100%;-webkit-background-clip:text;background-clip:text;color:transparent;-webkit-text-fill-color:transparent;font-family:'Montserrat','Segoe UI',sans-serif;font-size:1.15rem;font-weight:800;animation:tp4Flow 7s ease-in-out infinite;">Balayage du taux de bruit (0 → 40 % des labels de la classe minoritaire, 5 graines)</span></div>

</div>
<style>@keyframes tp4Flow{0%,100%{background-position:0% 50%}50%{background-position:100% 50%}}@keyframes tp4Glow{0%,100%{filter:brightness(1)}50%{filter:brightness(1.18)}}</style>

In [ ]:
rates = [0.0, 0.05, 0.1, 0.2, 0.3, 0.4]
t0 = time.perf_counter()
jobs = [(n, r, rate, s) for n in top3 for r in (False, True) for rate in rates for s in (range(1) if rate == 0 else range(5))]
sweep = run_all(jobs)
U.save_results("noise_sweep_classical.csv", sweep)
print(f"{len(jobs)} entraînements en {time.perf_counter() - t0:.0f}s")

agg = sweep.groupby(["model", "config", "noise_rate"])[["accuracy", "specificity", "roc_auc"]].agg(["mean", "std"]).reset_index()
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharex=True)
for ax, met, lab in zip(axes, ["accuracy", "specificity", "roc_auc"], ["test accuracy", "test specificity (normal recall)", "test ROC-AUC"]):
    for n in top3:
        for cfg, ls in [("baseline", "-"), ("robust", "--"), ("recalibrated", ":")]:
            d = agg[(agg.model == n) & (agg.config == cfg)]
            ax.errorbar(100 * d.noise_rate, d[(met, "mean")], yerr=d[(met, "std")], ls=ls, marker="o", ms=3,
                        color=COLOR[n], capsize=2, lw=1.6, label=f"{n} ({cfg})")
    ax.set(xlabel="% of minority labels flipped", ylabel=lab)
axes[0].legend(fontsize=6.5, ncol=1)
fig.suptitle("Classical models under label noise (solid: baseline, dashed: more regularised, dotted: threshold recalibrated)", y=1.02)
plt.tight_layout()
U.save_fig(fig, "fig_noise_sweep_classical")
plt.show()

<div style="background:rgba(142,36,170,.07);border-left:4px solid #8e24aa;border-radius:10px;padding:12px 16px;margin:6px 0;font-family:'Segoe UI',system-ui,sans-serif;"><b style="color:#8e24aa;">🔬 Analyse des résultats</b><div style="margin-top:6px;line-height:1.55;">
<ul style="margin:4px 0 0 0;">
<li><b>À 5 % (61 labels inversés)</b>, la chute est faible : environ −1 point d'accuracy et −2 à −4 points de
spécificité pour les baselines. Le F1, dominé par la classe Pneumonie majoritaire, bouge à peine : c'est un exemple de
plus où une seule métrique globale masque le problème.</li>
<li><b>Le bruit déplace le seuil, pas le classement.</b> En montant jusqu'à 40 %, la spécificité des baselines
s'effondre (≈ 0,6 → 0,2-0,3) et l'accuracy tombe vers 0,70-0,74, alors que l'<b>AUC reste quasi constante</b>
(0,92-0,95) : les modèles ordonnent toujours aussi bien les patients, mais ils ont appris une prévalence de
pneumonie trop élevée et déclarent « malade » trop facilement.</li>
<li><b>Plus de régularisation ne protège pas</b> : les variantes « robustes » font <b>systématiquement moins bien</b>
que les baselines, à tous les taux de bruit. Un modèle plus contraint s'appuie davantage sur l'a priori de classe,
justement faussé par ce bruit asymétrique.</li>
<li><b>Recalibrer le seuil sur un petit jeu propre suffit</b> à neutraliser l'essentiel de l'effet : l'accuracy des trois
modèles reste entre ≈ 0,86 et 0,89 de 0 à 40 % de bruit. La contrepartie est de disposer d'un jeu de validation fiable.</li>
<li>Aucun de ces modèles ne sait <b>quels</b> labels sont douteux, ni dire « je ne sais pas » face à une image
inhabituelle : c'est ce qu'apporte l'approche évidentielle (Partie 4).</li>
</ul></div></div>

<div style="margin-top:6px;"><span style="display:inline-block;background:linear-gradient(90deg,#4facfe,#2962ff,#7c4dff,#2962ff,#4facfe);background-size:300% 100%;-webkit-background-clip:text;background-clip:text;color:transparent;-webkit-text-fill-color:transparent;font-family:'Montserrat','Segoe UI',sans-serif;font-size:1.55rem;font-weight:800;animation:tp4Flow 7s ease-in-out infinite;">📦 Task 3.3 : MLOps avec MLflow</span></div>
<style>@keyframes tp4Flow{0%,100%{background-position:0% 50%}50%{background-position:100% 50%}}@keyframes tp4Glow{0%,100%{filter:brightness(1)}50%{filter:brightness(1.18)}}</style>

<div style="background:linear-gradient(135deg,rgba(142,36,170,.10),rgba(41,98,255,.07) 60%,rgba(79,172,254,.06));border-left:5px solid #8e24aa;border-radius:12px;padding:4px 20px 10px 20px;margin:8px 0;box-shadow:0 4px 16px rgba(142,36,170,.10);font-family:'Segoe UI',system-ui,sans-serif;line-height:1.6;">
<div style="height:3px;margin:0 -20px 6px -20px;border-radius:3px;background:linear-gradient(90deg,#8e24aa,#2962ff,#4facfe,#8e24aa);background-size:300% 100%;animation:tp4Flow 7s ease-in-out infinite;"></div>

**MLflow** trace chaque *run* (paramètres, métriques, artefacts, modèle) dans une **expérience**. Ici :

* le *tracking store* est local au projet (`mlflow.db`, artefacts dans `mlruns/`) — pas de chemin absolu propre à une machine ;
* on logge **tous les runs de la Task 3.2** (3 modèles × {baseline, robuste} × {propre, bruité}) avec leurs hyperparamètres
  (`C` pour les SVM / LogReg, `max_depth`, `K`…), le taux de bruit et les métriques (Precision, Recall, F1, …) ;
* on logge aussi les baselines de la Task 3.1 (dont **K du KNN** et **depth de l'arbre**) ;
* chaque modèle est enregistré comme **artefact** avec sa *signature* (schéma d'entrée/sortie) et le **modèle final**
  est ajouté au **Model Registry**, puis rechargé pour prouver qu'il est réutilisable.

</div>
<style>@keyframes tp4Flow{0%,100%{background-position:0% 50%}50%{background-position:100% 50%}}@keyframes tp4Glow{0%,100%{filter:brightness(1)}50%{filter:brightness(1.18)}}</style>

In [ ]:
import logging, os
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"   # pas de barres de progression dans le notebook
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.pipeline import make_pipeline
import warnings
logging.getLogger("mlflow").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="IProgress not found")
# Format « skops » (désérialisation sûre, par défaut dans MLflow 3) : on déclare explicitement comme fiable le seul
# types d'arbres non standards (HistGradientBoosting, DecisionTree/AdaBoost) que nous avons entraînés nous-mêmes.
TRUSTED_TYPES = ["sklearn.ensemble._hist_gradient_boosting.predictor.TreePredictor", "sklearn.tree._tree.Tree"]

mlflow.set_tracking_uri(f"sqlite:///{(U.ROOT / 'mlflow.db').as_posix()}")
EXPERIMENT = "TP4_PneumoniaMNIST_Robustness"
if mlflow.get_experiment_by_name(EXPERIMENT) is None:
    mlflow.create_experiment(EXPERIMENT, artifact_location=(U.ROOT / "mlruns").as_uri())
mlflow.set_experiment(EXPERIMENT)
print("MLflow", mlflow.__version__, "| tracking URI :", mlflow.get_tracking_uri())

In [ ]:
def log_run(name, robust, rate, seed=0, task="3.2"):
    y_fit = y_train if rate == 0 else U.flip_minority_labels(y_train, rate, seed=seed)[0]
    pipe = make_pipeline(StandardScaler(), make_model(name, robust)).fit(X_train, y_fit)   # le scaler voyage avec le modèle
    met = U.binary_metrics(y_test, pipe.predict(X_test), U.scores(pipe, X_test))
    est = pipe[-1]
    hp = {k: v for k, v in est.get_params().items()
          if k in {"C", "gamma", "kernel", "max_depth", "min_samples_leaf", "n_neighbors", "n_estimators",
                   "learning_rate", "max_leaf_nodes", "max_iter", "l2_regularization"} and not callable(v)}
    if name == "AdaBoost":
        hp["base_max_depth"] = est.estimator.max_depth
    run_name = f"{name}-{'robust' if robust else 'baseline'}-noise{int(100 * rate)}"
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tags({"task": task, "model_family": name, "config": "robust" if robust else "baseline",
                         "dataset": "PneumoniaMNIST-28", "positive_class": "pneumonia"})
        mlflow.log_params({**hp, "noise_rate": rate, "noise_seed": seed, "noise_type": "flip minority (normal->pneumonia)"})
        mlflow.log_metrics(met)
        mlflow.sklearn.log_model(pipe, name="model", signature=infer_signature(X_test[:5], pipe.predict(X_test[:5])),
                                 input_example=X_test[:2], skops_trusted_types=TRUSTED_TYPES)
    return {"run_id": run.info.run_id, "run_name": run_name, **met}

t0 = time.perf_counter()
logged = [log_run(n, r, rate) for n in top3 for r in (False, True) for rate in (0.0, 0.05)]
logged += [log_run(n, False, 0.0, task="3.1") for n in ["KNN", "Decision tree"] if n not in top3]
print(f"{len(logged)} runs loggés en {time.perf_counter() - t0:.0f}s")
pd.DataFrame(logged)[["run_name", "precision", "recall", "f1", "roc_auc"]].round(4)

In [ ]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT], filter_string="tags.task = '3.2' and params.noise_rate = '0.05'",
                          order_by=["metrics.f1 DESC"])
final = runs.iloc[0]
print(f"Modèle final (meilleur F1 test parmi les modèles entraînés sur labels bruités) : {final['tags.mlflow.runName']}")
reg = mlflow.register_model(f"runs:/{final.run_id}/model", "tp4-pneumonia-robust-classifier")
reloaded = mlflow.sklearn.load_model(f"models:/tp4-pneumonia-robust-classifier/{reg.version}")
print(f"Enregistré dans le Model Registry (version {reg.version}) ; rechargé → accuracy test = {accuracy_score(y_test, reloaded.predict(X_test)):.4f}")
cols = ["tags.mlflow.runName", "params.noise_rate", "params.C", "params.max_depth", "params.n_neighbors", "metrics.precision", "metrics.recall", "metrics.f1"]
all_runs = mlflow.search_runs(experiment_names=[EXPERIMENT])
all_runs[[c for c in cols if c in all_runs.columns]].round(4)

<div style="background:rgba(142,36,170,.07);border-left:4px solid #8e24aa;border-radius:10px;padding:12px 16px;margin:6px 0;font-family:'Segoe UI',system-ui,sans-serif;"><b style="color:#8e24aa;">🖥️ Visualiser les runs</b><div style="margin-top:6px;line-height:1.55;">
Dans un terminal, à la racine du projet :<br>
<code>.venv\Scripts\mlflow ui --backend-store-uri sqlite:///mlflow.db</code><br>
puis ouvrir <code>http://127.0.0.1:5000</code> : on peut comparer les runs (paramètres, métriques), télécharger les
artefacts et voir la version enregistrée du modèle final dans l'onglet <i>Models</i>.</div></div>

<div style="text-align:center;padding:18px 0;"><span style="display:inline-block;background:linear-gradient(90deg,#87ceeb,#1e90ff,#0047ab,#1e90ff,#87ceeb);background-size:300% 100%;-webkit-background-clip:text;background-clip:text;color:transparent;-webkit-text-fill-color:transparent;font-family:'Montserrat','Segoe UI',sans-serif;font-size:34px;font-weight:900;animation:tp4Flow 7s ease-in-out infinite;padding:14px 30px;border:2px solid rgba(30,144,255,.5);border-radius:12px;animation:tp4Flow 5s ease-in-out infinite,tp4Glow 3s ease-in-out infinite;">Merci de nous avoir écouté pour cette troisième partie !</span></div>
<style>@keyframes tp4Flow{0%,100%{background-position:0% 50%}50%{background-position:100% 50%}}@keyframes tp4Glow{0%,100%{filter:brightness(1)}50%{filter:brightness(1.18)}}</style>